# Custom CNN for MNIST Digit Classification

A CNN built from scratch in PyTorch to classify handwritten digits (0–9) from the MNIST dataset — 60,000 training images and 10,000 test images, each 28×28 pixels in grayscale.

**Pipeline:** Imports → Data loading → Model definition → Training → Evaluation → Inference

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Cell 2 — Hyperparameters & Data Loading

### Hyperparameters
- **`BATCH_SIZE = 64`** — Images processed per training step.
- **`LEARNING_RATE = 0.001`** — Standard starting value for Adam.
- **`EPOCHS = 3`** — Full passes over the training data.

### Device
Uses a GPU if CUDA is available, otherwise falls back to CPU.

### Transforms
1. **`ToTensor()`** — Converts PIL images to float tensors with values in `[0, 1]`.
2. **`Normalize((0.1307,), (0.3081,))`** — Standardizes using MNIST's global mean and std, centering pixel values around 0. Helps the model converge faster.

### DataLoader
`shuffle=True` on training data prevents the model from memorising the sample order. Test data stays unshuffled for consistent evaluation.

> **Bug fixed:** original code had `ATCH_SIZE = 64` (missing `B`), which caused a `NameError` in the DataLoader call.

In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

## Cell 3 — CNN Architecture

### Layers

**Convolutional layers** scan the image with small filters to detect spatial features like edges and curves:
- **`conv1`**: `Conv2d(1 → 32, kernel=3, padding=1)` — 1 grayscale input channel → 32 feature maps.
- **`conv2`**: `Conv2d(32 → 64, kernel=3, padding=1)` — 32 → 64 feature maps, learning higher-level patterns.

**`pool`**: `MaxPool2d(2, 2)` — Halves spatial dimensions. Applied after both conv layers.

**Feature-map SVD** is applied *after* each pooling layer. The pooled output tensor is reshaped into a 2D matrix,
low-rank approximated using SVD (keeping top-k singular values), then reshaped back. This compresses the
information flowing through the network without touching any learnable parameters.

**Fully connected layers:**
- **`fc1`**: `Linear(64×7×7 → 128)` — Flattens the (SVD-approximated) feature maps, compresses to 128.
- **`fc2`**: `Linear(128 → 10)` — Outputs 10 raw logits, one per digit class. **Not compressed by SVD.**

### Data flow through the network

```
Input  (B, 1, 28, 28)
  → conv1 + ReLU + pool  →  (B, 32, 14, 14)  → [SVD feature-map compression, k1]
  → conv2 + ReLU + pool  →  (B, 64,  7,  7)  → [SVD feature-map compression, k2]
  → flatten              →  (B, 3136)
  → fc1 + ReLU           →  (B, 128)
  → fc2                  →  (B, 10)   ← logits
```

No softmax at the end — `CrossEntropyLoss` applies it internally during training.

> **ReLU** (`max(0, x)`) is the activation function applied after each conv layer.


In [ ]:
import numpy as np

def svd_feature_map(x, k):
   
    if k is None:
        return x
    B, C, H, W = x.shape
    device = x.device
    out = torch.zeros_like(x)
    for b in range(B):
        # reshape (C, H, W) → (C, H*W)  — treat each channel as a row
        mat = x[b].view(C, H * W).cpu().detach().numpy()   # (C, H*W)
        U, S, Vt = np.linalg.svd(mat, full_matrices=False)
        k_eff = min(k, len(S))
        mat_approx = (U[:, :k_eff] * S[:k_eff]) @ Vt[:k_eff, :]  # (C, H*W)
        out[b] = torch.tensor(mat_approx, dtype=torch.float32).view(C, H, W).to(device)
    return out


class CustomCNN(nn.Module):
    def __init__(self, svd_k1=None, svd_k2=None):
        """
        svd_k1 : singular values kept after pool1  (None = disabled)
        svd_k2 : singular values kept after pool2  (None = disabled)
        """
        super(CustomCNN, self).__init__()
        self.conv1  = nn.Conv2d(in_channels=1,  out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2  = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool   = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1    = nn.Linear(64 * 7 * 7, 128)
        self.fc2    = nn.Linear(128, 10)
        self.svd_k1 = svd_k1   # k for feature-map SVD after pool1
        self.svd_k2 = svd_k2   # k for feature-map SVD after pool2

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # (B, 32, 14, 14)
        x = svd_feature_map(x, self.svd_k1)    # feature-map SVD after pool1
        x = self.pool(F.relu(self.conv2(x)))   # (B, 64,  7,  7)
        x = svd_feature_map(x, self.svd_k2)    # feature-map SVD after pool2
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


## Cell 4 — Instantiate the Model

Creates a `CustomCNN` instance and moves all its parameters to the target device with `.to(device)`. Any input data fed to the model must also be on the same device — this is done in the training loop.

In [ ]:
model = CustomCNN().to(device)
print(model)

## Cell 5 — Parameters & FLOPs

Before training, it's useful to know the size and cost of the model.

**Parameters** are all the learnable numbers in the model — every weight and bias across every layer. More parameters = more capacity to learn, but also more memory usage.

**FLOPs (Floating Point Operations)** count every multiply and add the model does to process one input image. This tells you the computational cost of a single forward pass — a proxy for inference speed.

`torchinfo.summary` gives a per-layer breakdown of both. `thop.profile` gives the total FLOPs. Both need a dummy input of the same shape as a real input — `(1, 1, 28, 28)` means 1 image, 1 channel, 28×28 pixels.

In [ ]:
!pip install torchinfo thop -q

from torchinfo import summary
from thop import profile

dummy_input = torch.zeros(1, 1, 28, 28).to(device)

print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
summary(model, input_size=(1, 1, 28, 28))

flops, params = profile(model, inputs=(dummy_input,), verbose=False)
print(f"\nTotal Parameters : {params:,.0f}")
print(f"Total FLOPs      : {flops:,.0f}  ({flops/1e6:.3f} MFLOPs)")

## Cell 6 — Loss Function & Optimizer

### `CrossEntropyLoss`
Standard loss for multi-class classification. It combines a softmax + negative log-likelihood internally, so the model just needs to output raw logits. The loss is high when the model is confidently wrong and approaches 0 as predictions improve.

### `Adam` optimizer
Adaptive gradient descent that maintains a per-parameter learning rate adjusted over time using momentum. Generally converges faster than plain SGD with less hyperparameter tuning. `model.parameters()` passes all trainable weights and biases to it.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Cell 7 — Training Function

`train()` runs one full epoch over all batches. The core loop follows the standard PyTorch training pattern:

| Step | What happens |
|------|--------------|
| `zero_grad()` | Clears accumulated gradients from the previous batch |
| `model(data)` | Forward pass — produces predictions |
| `criterion(...)` | Computes how far predictions are from true labels |
| `loss.backward()` | Backprop — computes gradient of loss w.r.t. every parameter |
| `optimizer.step()` | Updates weights using those gradients |

Progress is logged every 200 batches so training isn't silent for 3 epochs.

In [ ]:
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 200 == 0:
            print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} "
                  f"({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}")

## Cell 8 — Evaluation Function

Runs the model on the test set without updating any weights. Now returns accuracy so the training loop can track it per epoch.

- **`model.eval()`** — Switches off training-only behaviours (e.g. dropout).
- **`torch.no_grad()`** — Skips gradient computation entirely since we only need predictions, not backprop. Saves memory and speeds things up.

`output.argmax(dim=1)` picks the class index with the highest logit — that's the model's predicted digit.

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f"\nTest Set: Average loss: {test_loss:.4f}, "
          f"Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n")
    return accuracy

## Cell 9 — Training Loop

Runs `train()` followed by `test()` for each epoch, tracking accuracy per epoch. After 3 epochs on MNIST this model typically hits **~98–99% test accuracy**.

In [ ]:
accuracy_per_epoch = []

for epoch in range(1, EPOCHS + 1):
    train(model, device, train_loader, optimizer, criterion, epoch)
    acc = test(model, device, test_loader, criterion)
    accuracy_per_epoch.append(acc)

In [ ]:
# Google Drive save/load removed


In [ ]:
# Google Drive save/load removed


## Cell 10 — Final Summary

Prints everything in one place after training is done — parameters, FLOPs, and accuracy per epoch alongside peak accuracy. This gives a complete picture of the model: how big it is, how expensive it is to run, and how well it performs.

In [ ]:
flops, params = profile(model, inputs=(dummy_input,), verbose=False)

print("=" * 45)
print("           FINAL MODEL REPORT")
print("=" * 45)
print(f"  Total Parameters  : {params:,.0f}")
print(f"  Total FLOPs       : {flops/1e6:.3f} MFLOPs")
print("-" * 45)
for i, acc in enumerate(accuracy_per_epoch, 1):
    print(f"  Epoch {i} Accuracy  : {acc:.2f}%")
print("-" * 45)
print(f"  Peak Accuracy     : {max(accuracy_per_epoch):.2f}%")
print("=" * 45)

## Cell 9 — Prediction

Grabs a sample from the test set by index, shows the image, and prints what digit the model thinks it is. Change `idx` to any number between 0 and 9999 to try a different sample.

In [ ]:
import matplotlib.pyplot as plt

idx = 42
image, label = test_dataset[idx]

model.eval()
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    predicted = output.argmax(dim=1).item()

plt.imshow(image.squeeze(), cmap='gray')
plt.axis('off')
plt.title(f"Predicted: {predicted}")
plt.show()

## Cell 11 — SVD Setup

We apply SVD compression in **two distinct ways**:

**1. Weight compression on fc1 only**
The fc1 weight matrix W (shape 128 × 3136) is decomposed as W = UΣVᵀ. We keep only the top-k singular values
and reconstruct an approximated weight matrix that is plugged back in-place. fc2 is intentionally left untouched.

**2. Feature-map compression after pool1 and pool2**
This is *activation* compression, not parameter compression. After each pooling layer the output tensor
(B, C, H, W) is reshaped to (C, H×W) per sample, SVD is applied, and the rank-k approximation is reshaped back.
The conv weights themselves are unchanged — only the information flowing forward is compressed.

A drop of less than 0.5% from baseline accuracy is considered acceptable for all compression types.


In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

import copy
import matplotlib.pyplot as plt

def apply_svd_to_fc1(model, k):
    """In-place SVD approximation of fc1 weights only. fc2 is untouched."""
    W = model.fc1.weight.data.cpu().numpy()
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    W_approx = (U[:, :k] * S[:k]) @ Vt[:k, :]
    model.fc1.weight.data = torch.tensor(W_approx, dtype=torch.float32).to(device)

def evaluate_accuracy(model, device, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
    return 100. * correct / len(test_loader.dataset)

baseline_acc = evaluate_accuracy(model, device, test_loader)
print(f"Baseline accuracy (no compression): {baseline_acc:.2f}%")
print(f"Max possible k for fc1: {min(model.fc1.weight.shape)}")
print(f"Max possible k for feature-map SVD — after pool1: {14} (H=W=14), after pool2: {7} (H=W=7)")


## Cell 12 — Sweep k Values

**Part A — fc1 weight SVD sweep:** For each k we deep-copy the trained model, apply SVD to fc1 weights only
(fc2 untouched), and measure accuracy. Plots accuracy vs k and finds the minimum k within 0.5% of baseline.

**Part B — Feature-map SVD sweep:** We separately sweep k for the feature maps after pool1 and pool2.
For each k, a fresh copy of the model is created with `svd_k1=k` and `svd_k2=k` so SVD is applied
during the forward pass. Since the spatial dimensions are small (14×14 and 7×7), the maximum useful k
is 14 and 7 respectively.


In [ ]:
# ── Part A: fc1 weight SVD sweep ────────────────────────────────────────────
k_values = list(range(1, min(model.fc1.weight.shape), 5)) + [min(model.fc1.weight.shape)]
accuracies = []
threshold = baseline_acc - 0.5

for k in k_values:
    model_copy = copy.deepcopy(model)
    apply_svd_to_fc1(model_copy, k)
    acc = evaluate_accuracy(model_copy, device, test_loader)
    accuracies.append(acc)
    print(f"[fc1 SVD] k={k:4d} | Accuracy: {acc:.2f}%")

plt.figure(figsize=(10, 4))
plt.plot(k_values, accuracies, marker="o", markersize=3)
plt.axhline(y=threshold,    color="r", linestyle="--", label=f"Threshold ({threshold:.2f}%)")
plt.axhline(y=baseline_acc, color="g", linestyle="--", label=f"Baseline ({baseline_acc:.2f}%)")
plt.xlabel("k (singular values kept in fc1)")
plt.ylabel("Test Accuracy (%)")
plt.title("fc1 Weight SVD — Accuracy vs k")
plt.legend()
plt.grid(True)
plt.show()

min_k = next((k for k, acc in zip(k_values, accuracies) if acc >= threshold), None)
print(f"\nMinimum k (fc1) to stay within 0.5% of baseline: {min_k}")


# ── Part B: feature-map SVD sweep (pool1 and pool2 together) ─────────────────
# Max k is bounded by H*W of the feature map: 14 for pool1, 7 for pool2.
# We use the same k for both layers in this sweep for simplicity.
fm_k_values = list(range(1, 8)) + [14]   # up to max of pool2 (7), plus pool1 max (14)
fm_accuracies = []

print("\n" + "-"*55)
print("Feature-map SVD sweep (applied after pool1 and pool2)")
print("-"*55)
for k in fm_k_values:
    # k2 capped at 7 (pool2 spatial size); k1 uses same k (capped at 14 by svd_feature_map)
    k2 = min(k, 7)
    model_copy = copy.deepcopy(model)
    model_copy.svd_k1 = k
    model_copy.svd_k2 = k2
    acc = evaluate_accuracy(model_copy, device, test_loader)
    fm_accuracies.append(acc)
    print(f"[FM SVD]  k1={k:2d}, k2={k2:2d} | Accuracy: {acc:.2f}%")

plt.figure(figsize=(10, 4))
plt.plot(fm_k_values, fm_accuracies, marker="s", color="darkorange", markersize=5)
plt.axhline(y=threshold,    color="r", linestyle="--", label=f"Threshold ({threshold:.2f}%)")
plt.axhline(y=baseline_acc, color="g", linestyle="--", label=f"Baseline ({baseline_acc:.2f}%)")
plt.xlabel("k (singular values kept in feature maps)")
plt.ylabel("Test Accuracy (%)")
plt.title("Feature-map SVD — Accuracy vs k (after pool1 & pool2)")
plt.legend()
plt.grid(True)
plt.show()

min_k_fm = next((k for k, acc in zip(fm_k_values, fm_accuracies) if acc >= threshold), None)
print(f"\nMinimum k (feature-map) to stay within 0.5% of baseline: {min_k_fm}")


## Cell 13 — Apply Final Compression at Best k

Using the minimum acceptable k values found in Cell 12, we apply both compression types and print side-by-side reports:
- **fc1 weight SVD**: in-place low-rank approximation of the fc1 weight matrix. fc2 is untouched.
- **Feature-map SVD**: forward-pass activation compression after pool1 and pool2. No weights change.


In [ ]:
# ── fc1 weight compression report ────────────────────────────────────────────
best_k = min_k

compressed_model = copy.deepcopy(model)
apply_svd_to_fc1(compressed_model, best_k)
compressed_acc = evaluate_accuracy(compressed_model, device, test_loader)

fc1_orig = model.fc1.weight.shape[0] * model.fc1.weight.shape[1]
fc1_comp = best_k * (model.fc1.weight.shape[0] + model.fc1.weight.shape[1])

print("=" * 50)
print("     fc1 WEIGHT SVD COMPRESSION REPORT")
print("=" * 50)
print(f"  Best k (fc1)            : {best_k}")
print("-" * 50)
print(f"  fc1 original params     : {fc1_orig:,}")
print(f"  fc1 compressed params   : {fc1_comp:,}")
print(f"  fc1 reduction           : {100*(1 - fc1_comp/fc1_orig):.1f}%")
print("-" * 50)
print(f"  fc2                     : unchanged (not compressed)")
print("-" * 50)
print(f"  Baseline accuracy       : {baseline_acc:.2f}%")
print(f"  Compressed accuracy     : {compressed_acc:.2f}%")
print(f"  Accuracy drop           : {baseline_acc - compressed_acc:.2f}%")
print("=" * 50)

# ── feature-map compression report ───────────────────────────────────────────
best_k_fm = min_k_fm
best_k2_fm = min(best_k_fm, 7)

fm_model = copy.deepcopy(model)
fm_model.svd_k1 = best_k_fm
fm_model.svd_k2 = best_k2_fm
fm_acc = evaluate_accuracy(fm_model, device, test_loader)

print()
print("=" * 50)
print("   FEATURE-MAP SVD COMPRESSION REPORT")
print("=" * 50)
print(f"  k used after pool1      : {best_k_fm}  (max possible: 14)")
print(f"  k used after pool2      : {best_k2_fm}  (max possible:  7)")
print("-" * 50)
print(f"  Note: No parameters are changed — this compresses")
print(f"        the activation tensors in the forward pass.")
print("-" * 50)
print(f"  Baseline accuracy       : {baseline_acc:.2f}%")
print(f"  FM-compressed accuracy  : {fm_acc:.2f}%")
print(f"  Accuracy drop           : {baseline_acc - fm_acc:.2f}%")
print("=" * 50)


## Cell 13b — Actual Layer Restructuring of fc1 (Real Parameter Compression)

Now we **actually restructure fc1** into two smaller Linear layers using SVD. fc2 is left completely unchanged.

**How it works:**
W = U · diag(S) · Vᵀ splits one big matrix multiply into two smaller ones:
```
output = W @ input
       = (U · diag(S)) @ (Vt @ input)
       = layer_A @ (layer_B @ input)
```
So `Linear(3136, 128)` becomes:
- `Linear(3136, k, bias=False)` — applies Vᵀ (projects input to k dims)
- `Linear(k, 128)` — applies U·S (maps k dims to output)

**Parameter count:**
```
Original fc1:    128 × 3136              = 401,408
Restructured:    3136×k + k×128 + 128   = k×(3136+128) + 128
```
fc2 (128×10 = 1,280 params) stays exactly as is.


In [ ]:
class SVDLinear(nn.Module):
    """Replaces a Linear layer with two smaller layers via SVD decomposition."""
    def __init__(self, in_features, out_features, k, bias=True):
        super().__init__()
        self.layer_b = nn.Linear(in_features, k, bias=False)
        self.layer_a = nn.Linear(k, out_features, bias=bias)

    def forward(self, x):
        return self.layer_a(self.layer_b(x))


def restructure_fc1_with_svd(model, k_fc1):
    """Restructure fc1 only into two SVDLinear layers. fc2 is untouched."""
    restructured = copy.deepcopy(model)

    W1 = model.fc1.weight.data.cpu().numpy()
    U1, S1, Vt1 = np.linalg.svd(W1, full_matrices=False)

    Vt1_k = Vt1[:k_fc1, :]              # (k, 3136)
    US1_k = U1[:, :k_fc1] * S1[:k_fc1]  # (128, k)

    svd_fc1 = SVDLinear(model.fc1.in_features, model.fc1.out_features, k_fc1)
    svd_fc1.layer_b.weight.data = torch.tensor(Vt1_k, dtype=torch.float32)
    svd_fc1.layer_a.weight.data = torch.tensor(US1_k, dtype=torch.float32)
    if model.fc1.bias is not None:
        svd_fc1.layer_a.bias.data = model.fc1.bias.data.clone()
    restructured.fc1 = svd_fc1

    # fc2 is NOT touched
    return restructured.to(device)


real_compressed = restructure_fc1_with_svd(model, k_fc1=min_k)
real_acc = evaluate_accuracy(real_compressed, device, test_loader)

orig_params = count_params(model)
real_params = count_params(real_compressed)

print("=" * 55)
print("     REAL SVD RESTRUCTURING REPORT (fc1 only)")
print("=" * 55)
print(f"  k used (fc1)            : {min_k}")
print("-" * 55)
print(f"  Original total params   : {orig_params:,}")
print(f"  Restructured params     : {real_params:,}")
print(f"  Actual reduction        : {100*(1 - real_params/orig_params):.1f}%")
print("-" * 55)
print(f"  Baseline accuracy       : {baseline_acc:.2f}%")
print(f"  Restructured accuracy   : {real_acc:.2f}%")
print(f"  Accuracy drop           : {baseline_acc - real_acc:.2f}%")
print("=" * 55)
print()
print("  fc1 is now:", real_compressed.fc1)
print("  fc2 is now:", real_compressed.fc2, "  ← unchanged")

# --- memory and FLOPs (fc1 only) ---
bytes_per_param = 4

fc1_orig_params = model.fc1.weight.numel()
fc1_comp_params = real_compressed.fc1.layer_b.weight.numel() + real_compressed.fc1.layer_a.weight.numel()

mem_orig_kb = fc1_orig_params * bytes_per_param / 1024
mem_comp_kb = fc1_comp_params * bytes_per_param / 1024

flops_fc1_orig = 2 * model.fc1.in_features * model.fc1.out_features
flops_fc1_comp = (2 * real_compressed.fc1.layer_b.in_features * real_compressed.fc1.layer_b.out_features +
                  2 * real_compressed.fc1.layer_a.in_features * real_compressed.fc1.layer_a.out_features)

print()
print("=" * 55)
print("     MEMORY & COMPUTE SAVINGS (fc1)")
print("=" * 55)
print(f"  fc1 memory (original)   : {mem_orig_kb:.1f} KB")
print(f"  fc1 memory (restructured): {mem_comp_kb:.1f} KB")
print(f"  Memory saved            : {mem_orig_kb - mem_comp_kb:.1f} KB  ({100*(1 - mem_comp_kb/mem_orig_kb):.1f}% reduction)")
print("-" * 55)
print(f"  fc1 FLOPs (original)    : {flops_fc1_orig:,}")
print(f"  fc1 FLOPs (restructured): {flops_fc1_comp:,}")
print(f"  FLOPs saved             : {flops_fc1_orig - flops_fc1_comp:,}  ({100*(1 - flops_fc1_comp/flops_fc1_orig):.1f}% reduction)")
print("=" * 55)


## Cell 14 — ARSVD (Adaptive Rank SVD)

Regular SVD (Cells 11–13) required you to manually sweep k values and pick one. ARSVD automates this — it finds the right k **per layer** using **spectral entropy**.

**The idea behind spectral entropy:**
After SVD, normalise the singular values into a probability distribution:
```
pᵢ = sᵢ / (s₁ + s₂ + ... + sᵣ)
```
Then compute entropy:
```
H_total = -Σ pᵢ log(pᵢ)
```
Low entropy = information concentrated in a few singular values = compress aggressively.
High entropy = information spread across many = need more singular values to preserve the layer.

ARSVD scans from k=1 upward and stops at the smallest k where the partial entropy `H(k)` reaches `τ × H_total`. τ is your only input — one number controls compression across all layers.

Each layer gets its own k automatically — no manual tuning per layer needed.

In [ ]:
def arsvd(W, tau=0.9):
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    r = len(S)

    # normalise singular values into a probability distribution
    p = S / S.sum()

    # total spectral entropy of this layer
    H_total = -np.sum(p * np.log(p + 1e-12))

    # scan k from 1 upward, stop when partial entropy hits tau * H_total
    H_k = 0.0
    k = r
    for j in range(r):
        H_k += -p[j] * np.log(p[j] + 1e-12)
        if H_k >= tau * H_total:
            k = j + 1
            break

    W_compressed = (U[:, :k] * S[:k]) @ Vt[:k, :]
    return W_compressed, k


def apply_arsvd_to_model(model, tau=0.9):
    compressed = copy.deepcopy(model)
    results = {}

    # Only compress fc1 — fc2 is deliberately excluded
    W = compressed.fc1.weight.data.cpu().numpy()
    W_comp, k = arsvd(W, tau)
    compressed.fc1.weight.data = torch.tensor(W_comp, dtype=torch.float32).to(device)
    results['fc1'] = {
        'original_params': W.shape[0] * W.shape[1],
        'compressed_params': k * (W.shape[0] + W.shape[1]),
        'k': k,
        'max_k': min(W.shape)
    }

    # fc2 stays exactly as trained
    results['fc2'] = {
        'original_params': compressed.fc2.weight.numel(),
        'compressed_params': compressed.fc2.weight.numel(),  # unchanged
        'k': 'N/A (not compressed)',
        'max_k': min(compressed.fc2.weight.shape)
    }

    return compressed, results


## Cell 15 — Run ARSVD at Different τ Values

τ controls how much of the spectral information each layer must retain:
- τ = 0.99 → very conservative, keeps most singular values, minimal accuracy drop
- τ = 0.90 → moderate compression
- τ = 0.70 → aggressive compression, higher accuracy risk

We sweep τ from 0.5 to 0.99 and record what k each layer gets and what accuracy comes out. This lets you pick the τ that gives the best compression-accuracy tradeoff.

In [ ]:
tau_values = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
arsvd_results = []

orig_fc1_params = model.fc1.weight.numel()   # fc2 is not in the compression budget

print(f"{'tau':>6} | {'fc1 k':>6} | {'fc1 params':>12} | {'param red%':>10} | {'Accuracy':>10} | {'acc drop%':>10}")
print("-" * 70)

for tau in tau_values:
    comp_model, layer_info = apply_arsvd_to_model(model, tau)
    acc = evaluate_accuracy(comp_model, device, test_loader)

    fc1 = layer_info.get('fc1', {})
    comp_fc1_params = fc1.get('compressed_params', 0)
    param_red = 100 * (1 - comp_fc1_params / orig_fc1_params)
    acc_drop  = baseline_acc - acc

    arsvd_results.append({
        'tau':        tau,
        'fc1_k':      fc1.get('k', '-'),
        'fc1_params': comp_fc1_params,
        'param_red':  param_red,
        'accuracy':   acc,
        'acc_drop':   acc_drop
    })

    print(f"{tau:>6.2f} | {fc1.get('k',0):>6} | {comp_fc1_params:>12,} | "
          f"{param_red:>9.1f}% | {acc:>9.2f}% | {acc_drop:>9.2f}%")

# plot
param_reds = [r['param_red'] for r in arsvd_results]
acc_drops  = [r['acc_drop']  for r in arsvd_results]
taus       = [r['tau']       for r in arsvd_results]

fig, ax1 = plt.subplots(figsize=(9, 5))
color1 = 'steelblue'
color2 = 'tomato'

ax1.plot(taus, param_reds, marker='o', color=color1, label='fc1 Parameter Reduction %')
ax1.set_xlabel('tau')
ax1.set_ylabel('fc1 Parameter Reduction (%)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(taus, acc_drops, marker='s', color=color2, label='Accuracy Drop %')
ax2.set_ylabel('Accuracy Drop (%)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left')

plt.title('ARSVD (fc1 only): Parameter Reduction vs Accuracy Drop across tau')
plt.grid(True)
plt.tight_layout()
plt.show()


## Cell 16 — Final ARSVD Report

Picks the best τ — the smallest one where accuracy stays within 0.5% of baseline — and prints the full compression report.

Then prints a **comparison table** showing Baseline vs ARSVD at every τ value, with columns:
- **tau** — entropy threshold used (Baseline row = no compression)
- **Params** — total model parameters
- **FLOPs** — total floating point operations for one forward pass
- **Accuracy** — test set accuracy (%)

The best τ row is marked with an arrow.


In [ ]:
# pick smallest tau that keeps accuracy within 0.5% of baseline
best = next((r for r in arsvd_results if r['accuracy'] >= baseline_acc - 0.5), arsvd_results[-1])
best_tau = best['tau']

final_model, final_info = apply_arsvd_to_model(model, best_tau)
final_acc = evaluate_accuracy(final_model, device, test_loader)

fc1_info = final_info['fc1']

print("=" * 55)
print("        ARSVD COMPRESSION REPORT (fc1 only)")
print("=" * 55)
print(f"  Entropy threshold (tau) : {best_tau}")
print("-" * 55)
print(f"  fc1 — k selected        : {fc1_info['k']} / {fc1_info['max_k']} (max)")
print(f"  fc1 — original params   : {fc1_info['original_params']:,}")
print(f"  fc1 — compressed params : {fc1_info['compressed_params']:,}")
print(f"  fc1 — reduction         : {100*(1 - fc1_info['compressed_params']/fc1_info['original_params']):.1f}%")
print("-" * 55)
print(f"  fc2                     : unchanged (not compressed)")
print("-" * 55)
print(f"  Baseline accuracy       : {baseline_acc:.2f}%")
print(f"  ARSVD accuracy          : {final_acc:.2f}%")
print(f"  Accuracy drop           : {baseline_acc - final_acc:.2f}%")
print("=" * 55)
print()
print("  SVD vs ARSVD comparison (fc1):")
print(f"  SVD best k (fc1)        : {min_k}  (manually found by sweep)")
print(f"  ARSVD k (fc1)           : {fc1_info['k']}  (auto via spectral entropy, tau={best_tau})")


# ── Comparison Table: Baseline vs ARSVD at each tau ──────────────────────────
from thop import profile as thop_profile
import torch

dummy = torch.zeros(1, 1, 28, 28).to(device)

# baseline stats
baseline_params = count_params(model)
baseline_flops, _ = thop_profile(model, inputs=(dummy,), verbose=False)

# collect per-tau stats
table_rows = []
for r in arsvd_results:
    tau = r['tau']
    comp_model, _ = apply_arsvd_to_model(model, tau)
    comp_params = count_params(comp_model)
    comp_flops, _ = thop_profile(comp_model, inputs=(dummy,), verbose=False)
    table_rows.append({
        'tau':      tau,
        'params':   comp_params,
        'flops':    comp_flops,
        'accuracy': r['accuracy']
    })

# print table
print()
print("=" * 75)
print("          BASELINE vs ARSVD COMPARISON TABLE")
print("=" * 75)
print(f"  {'tau':>8} | {'Params':>12} | {'FLOPs':>14} | {'Accuracy':>10}")
print("-" * 75)
# baseline row
print(f"  {'Baseline':>8} | {baseline_params:>12,} | {baseline_flops:>14,.0f} | {baseline_acc:>9.2f}%")
print("-" * 75)
# one row per tau
for row in table_rows:
    marker = " <-- best" if row['tau'] == best_tau else ""
    print(f"  {row['tau']:>8.2f} | {row['params']:>12,} | {row['flops']:>14,.0f} | {row['accuracy']:>9.2f}%{marker}")
print("=" * 75)
